In [1]:
from pyspark.sql import SparkSession, Row
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, StringType, TimestampType, LongType, DoubleType
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.sql.functions import col, monotonically_increasing_id, lit, min, max
from pyspark.sql import functions as F
from sklearn.metrics import confusion_matrix, precision_score, recall_score, accuracy_score, f1_score, roc_auc_score
from pyspark.sql.functions import when, col, isnan, to_date, from_unixtime
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline

In [2]:
spark = SparkSession.builder.master('local[*]').appName("K-means Research").config('spark.ui.port','4050').getOrCreate()

26/08/26 20:19:47 WARN Utils: Your hostname, car-ubuntu resolves to a loopback address: 127.0.1.1; using 192.168.1.42 instead (on interface ens33)
26/08/26 20:19:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/26 20:19:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/26 20:19:49 WARN Utils: Service 'SparkUI' could not bind on port 4050. Attempting port 4051.
26/08/26 20:19:49 WARN Utils: Service 'SparkUI' could not bind on port 4051. Attempting port 4052.


In [3]:
file_path = "/home/ubuntu/Downloads/Zeek_Parquet"
df = spark.read.parquet(file_path, header = True, inferSchema = True)

In [4]:
df.count()

700340

In [5]:
df.select("*").where("label_binary = true").count()

346933

In [6]:
df.select("*").where("label_binary = false").count()

350339

In [7]:
df = df.fillna(0, subset=["duration"])

In [8]:
#df.select("duration").distinct().collect()

In [9]:
min_max_values = df.select(min("duration").alias("min_duration"), max("duration").alias("max_duration"))
min_max_values.show()

+------------+-----------------+
|min_duration|     max_duration|
+------------+-----------------+
|         0.0|3560.855576992035|
+------------+-----------------+



min max: all bytes pkts and duration, table with attributes on top min x max and paragraph ab scalar.

In [10]:
#df.printSchema()

In [11]:
df.select("ts",
    "proto",
    "service",
    "conn_state",
    "orig_bytes",
    "orig_ip_bytes",
    "orig_pkts",
    "resp_bytes",
    "resp_pkts",
    "resp_ip_bytes",
    "missed_bytes",
    "dest_port_zeek",
    "src_port_zeek").limit(5).toPandas()

,ts,proto,service,conn_state,orig_bytes,orig_ip_bytes,orig_pkts,resp_bytes,resp_pkts,resp_ip_bytes,missed_bytes,dest_port_zeek,src_port_zeek
0,1.663789e+09,tcp,None,S0,NaN,44,1,NaN,0,0,0,36197,44183
1,1.663789e+09,tcp,None,S0,NaN,44,1,NaN,0,0,0,8323,44184
2,1.663789e+09,tcp,None,S0,NaN,44,1,NaN,0,0,0,47158,44183
3,1.663789e+09,tcp,None,S0,NaN,44,1,NaN,0,0,0,34135,44183
4,1.663789e+09,tcp,None,S0,NaN,44,1,NaN,0,0,0,9838,44184


In [12]:
columns_to_check = [
    "ts",
    "proto",
    "service",
    "conn_state",
    "orig_bytes",
    "orig_ip_bytes",
    "orig_pkts",
    "resp_bytes",
    "resp_pkts",
    "resp_ip_bytes",
    "missed_bytes",
    "dest_port_zeek",
    "src_port_zeek"
]


In [13]:
df = df.na.fill(0, subset=columns_to_check)

In [14]:
df.select("ts",
    "proto",
    "service",
    "conn_state",
    "orig_bytes",
    "orig_ip_bytes",
    "orig_pkts",
    "resp_bytes",
    "resp_pkts",
    "resp_ip_bytes",
    "missed_bytes",
    "dest_port_zeek",
    "src_port_zeek").limit(6).toPandas()

,ts,proto,service,conn_state,orig_bytes,orig_ip_bytes,orig_pkts,resp_bytes,resp_pkts,resp_ip_bytes,missed_bytes,dest_port_zeek,src_port_zeek
0,1.663789e+09,tcp,None,S0,0,44,1,0,0,0,0,36197,44183
1,1.663789e+09,tcp,None,S0,0,44,1,0,0,0,0,8323,44184
2,1.663789e+09,tcp,None,S0,0,44,1,0,0,0,0,47158,44183
3,1.663789e+09,tcp,None,S0,0,44,1,0,0,0,0,34135,44183
4,1.663789e+09,tcp,None,S0,0,44,1,0,0,0,0,9838,44184
5,1.663789e+09,tcp,None,S0,0,44,1,0,0,0,0,62471,44184


## Please record how we did this one on the word file 

In [15]:
#Binning ports based on binning file
def bin_ports(df, column):
    return df.withColumn(
        column,
        when(col(column).isNull(), 0)
        .when((col(column) >= 0) & (col(column) <= 1023), 1)
        .when((col(column) >= 1024) & (col(column) <= 49151), 2)
        .when((col(column) >= 49152) & (col(column) <= 65535), 3)
        .otherwise(4)
    )

columns = ["dest_port_zeek", "src_port_zeek"]
for column in columns:
    df = bin_ports(df, column)

In [16]:
df.select("dest_port_zeek", "src_port_zeek").limit(10).toPandas()

,dest_port_zeek,src_port_zeek
0,2,2
1,2,2
2,2,2
3,2,2
4,2,2
5,3,2
6,2,3
7,2,3
8,2,2
9,2,2


### Bining the service column - first way, no need to be recorded.

In [17]:
##df = df.fillna({'service': 'unknown'})
#
#indexer = StringIndexer(inputCol='service', outputCol='service_index', handleInvalid='keep')
#
#encoder = OneHotEncoder(inputCols=['service_index'], outputCols=['service_vec'])
#
#pipeline = Pipeline(stages=[indexer, encoder])
#model = pipeline.fit(df)
#df = model.transform(df)
#
#df.select('service', 'service_index', 'service_vec').show()

### Bining protocol column

In [18]:
df.select("proto").limit(5).toPandas()

,proto
0,tcp
1,tcp
2,tcp
3,tcp
4,tcp


In [19]:
df = df.withColumn(
    'proto',
    when((col('proto').isin('tcp')) , 0)
        .when((col('proto').isin('udp')), 1)
        .when((col('proto').isin('icmp')), 2)
        .otherwise(3)
)
df.select("proto").limit(5).toPandas()

,proto
0,0
1,0
2,0
3,0
4,0


### Binning Conn_state column

In [20]:
conn_state_names = df.select("conn_state").distinct().collect()
conn_state_names
df = df.withColumn(
    'conn_state',
    when((col('conn_state').isin('RSTRH')) , 1)
        .when((col('conn_state').isin('SF')), 2)
        .when((col('conn_state').isin('S0')), 3)
        .when((col('conn_state').isin('OTH')), 4)
        .when((col('conn_state').isin('REJ')), 5)
        .when((col('conn_state').isin('RSTO')), 6)
        .when((col('conn_state').isin('RSTR')), 7)
        .when((col('conn_state').isin('SH')), 8)
        .when((col('conn_state').isin('S2')), 9)
        .when((col('conn_state').isin('S1')), 10)
        .when((col('conn_state').isin('SHR')), 11)
        .otherwise(0)
)
df.select("conn_state").limit(5).toPandas()

,conn_state
0,3
1,3
2,3
3,3
4,3


### Bining Service column - way that worked better please record!

In [21]:
df = df.withColumn(
    'service',
    when((col('service').isin('ftp')) , 1)
        .when((col('service').isin('unknown')), 2)
        .when((col('service').isin('http')), 3)
        .when((col('service').isin('ntlm,smb,gssapi')), 4)
        .when((col('service').isin('gssapi')), 5)
        .when((col('service').isin('gssapi,smb,dce_rpc,ntlm')), 6)
        .when((col('service').isin('smb,gssapi')), 7)
        .when((col('service').isin('dns')), 8)
        .when((col('service').isin('radius')), 9)
        .when((col('service').isin('ssl')), 10)
        .when((col('service').isin('ntp')), 11)
        .when((col('service').isin('dhcp')), 12)
        .when((col('service').isin('dce_rpc,smb,gssapi,ntlm')), 13)
        .when((col('service').isin('gssapi,smb')), 14)
        .when((col('service').isin('gssapi,smb,ntlm')), 15)
        .when((col('service').isin('ntlm,gssapi,dce_rpc,smb')), 16)
        .when((col('service').isin('smb,ntlm,gssapi')), 17)
        .when((col('service').isin('snmp')), 18)
        .otherwise(0)
)
df.select("service").limit(5).toPandas()

,service
0,0
1,0
2,0
3,0
4,0


In [22]:
df.select("ts",
          "duration",
    "proto",
    "service",
    "conn_state",
    "orig_bytes",
    "orig_ip_bytes",
    "orig_pkts",
    "resp_bytes",
    "resp_pkts",
    "resp_ip_bytes",
    "missed_bytes",
    "dest_port_zeek",
    "src_port_zeek").limit(6).toPandas()

,ts,duration,proto,service,conn_state,orig_bytes,orig_ip_bytes,orig_pkts,resp_bytes,resp_pkts,resp_ip_bytes,missed_bytes,dest_port_zeek,src_port_zeek
0,1.663789e+09,0.0,0,0,3,0,44,1,0,0,0,0,2,2
1,1.663789e+09,0.0,0,0,3,0,44,1,0,0,0,0,2,2
2,1.663789e+09,0.0,0,0,3,0,44,1,0,0,0,0,2,2
3,1.663789e+09,0.0,0,0,3,0,44,1,0,0,0,0,2,2
4,1.663789e+09,0.0,0,0,3,0,44,1,0,0,0,0,2,2
5,1.663789e+09,0.0,0,0,3,0,44,1,0,0,0,0,3,2


# K-means clustering

In [23]:
columns_to_check = ["duration", 
                    "resp_pkts", 
                    "conn_state",
                    "proto", 
                    "orig_ip_bytes",
                    "missed_bytes",
                    "orig_pkts",
                    "resp_ip_bytes", 
                    "dest_port_zeek", 
                    "orig_bytes", 
                    "resp_bytes", 
                    "src_port_zeek"
                   ] 
df_filled = df.na.fill(0, subset=columns_to_check)

assembler = VectorAssembler(
    inputCols = columns_to_check,
    outputCol = "features"
)
df_assembled = assembler.transform(df)
scaler = StandardScaler(inputCol = "features", outputCol = "scaledFeatures")
scaler_model = scaler.fit(df_assembled)
df_scaled = scaler_model.transform(df_assembled)
df_scaled.select("features", "scaledFeatures").limit(5).toPandas()

,features,scaledFeatures
0,"(0.0, 0.0, 3.0, 0.0, 44.0, 0.0, 1.0, 0.0, 2.0,...","(0.0, 0.0, 2.589315886884535, 0.0, 0.018898205..."
1,"(0.0, 0.0, 3.0, 0.0, 44.0, 0.0, 1.0, 0.0, 2.0,...","(0.0, 0.0, 2.589315886884535, 0.0, 0.018898205..."
2,"(0.0, 0.0, 3.0, 0.0, 44.0, 0.0, 1.0, 0.0, 2.0,...","(0.0, 0.0, 2.589315886884535, 0.0, 0.018898205..."
3,"(0.0, 0.0, 3.0, 0.0, 44.0, 0.0, 1.0, 0.0, 2.0,...","(0.0, 0.0, 2.589315886884535, 0.0, 0.018898205..."
4,"(0.0, 0.0, 3.0, 0.0, 44.0, 0.0, 1.0, 0.0, 2.0,...","(0.0, 0.0, 2.589315886884535, 0.0, 0.018898205..."


In [24]:
df_scaled_attacks = df_scaled.filter(col("label_binary") == True)
df_scaled_normalData = df_scaled.filter(col("label_binary") == False)

In [25]:
attack_means = df_scaled_attacks.select(
    *[mean(c).alias(c) for c in columns_to_check]
).toPandas().T

normal_means = df_scaled_normalData.select(
    *[mean(c).alias(c) for c in columns_to_check]
).toPandas().T

# Combine into one DataFrame
comparison = attack_means.join(normal_means, lsuffix="_attack", rsuffix="_normal")
comparison.columns = ["Attack Mean", "Normal Mean"]

# Optional: round to 3 decimal places
comparison = comparison.round(3)

print(comparison)

NameError: name 'mean' is not defined

| Feature            | Can an attacker influence it? | Notes                                                                                                                    |
| ------------------ | ----------------------------- | ------------------------------------------------------------------------------------------------------------------------ |
| **duration**       | **Yes**                       | An attacker can end a connection quickly or keep it open longer.                                                         |
| **resp_pkts**      | **Indirectly**                | The attacker can't directly choose how many packets the server sends, but their requests influence the response.         |
| **conn_state**     | **Yes (partially)**           | An attacker can intentionally reset connections, never complete the TCP handshake, or close connections in unusual ways. |
| **proto**          | **Yes**                       | The attacker chooses whether to use TCP, UDP, ICMP, etc.                                                                 |
| **orig_ip_bytes**  | **Yes**                       | The attacker controls how much data they send.                                                                           |
| **missed_bytes**   | **Usually No**                | This is a capture artifact (e.g., packets missed by the sensor), not something the attacker directly controls.           |
| **orig_pkts**      | **Yes**                       | The attacker controls how many packets they transmit.                                                                    |
| **resp_ip_bytes**  | **Indirectly**                | The server decides what to send, but the attacker's requests affect the response size.                                   |
| **dest_port_zeek** | **Yes**                       | The attacker chooses which destination port to target (22, 80, 443, etc.).                                               |
| **orig_bytes**     | **Yes**                       | Fully controlled by the attacker.                                                                                        |
| **resp_bytes**     | **Indirectly**                | Depends on the victim's response to the attacker's request.                                                              |
| **src_port_zeek**  | **Yes**                       | The attacker (or their operating system) chooses the source port.                                                        |


In [ ]:
from pyspark.sql.functions import percentile_approx

# Compute medians
attack_medians = df_scaled_attacks.select(
    *[percentile_approx(c, 0.5, 10000).alias(c) for c in columns_to_check]
).toPandas().T

normal_medians = df_scaled_normalData.select(
    *[percentile_approx(c, 0.5, 10000).alias(c) for c in columns_to_check]
).toPandas().T

# Combine into one DataFrame
comparison = attack_medians.join(
    normal_medians,
    lsuffix="_attack",
    rsuffix="_normal"
)

comparison.columns = ["Attack Median", "Normal Median"]

# Display with 5 decimal places (without changing the underlying values)
print(comparison.to_string(float_format="{:.5f}".format))

In [26]:
from pyspark.sql.functions import min, max
import pandas as pd

columns_to_check = [
    "duration",
    "resp_pkts",
    "conn_state",
    "proto",
    "orig_ip_bytes",
    "missed_bytes",
    "orig_pkts",
    "resp_ip_bytes",
    "dest_port_zeek",
    "orig_bytes",
    "resp_bytes",
    "src_port_zeek"
]

result = df_scaled.select(
    *[min(c).alias(f"{c}_min") for c in columns_to_check],
    *[max(c).alias(f"{c}_max") for c in columns_to_check]
).toPandas()

min_max_df = pd.DataFrame({
    "Column": columns_to_check,
    "Minimum": [result[f"{c}_min"][0] for c in columns_to_check],
    "Maximum": [result[f"{c}_max"][0] for c in columns_to_check]
})

print(min_max_df)

[Stage 28:==============>                                           (1 + 3) / 4]

            Column  Minimum       Maximum
0         duration      0.0  3.560856e+03
1        resp_pkts      0.0  1.004000e+04
2       conn_state      1.0  1.100000e+01
3            proto      0.0  2.000000e+00
4    orig_ip_bytes      0.0  9.597360e+05
5     missed_bytes      0.0  1.909000e+04
6        orig_pkts      0.0  1.004000e+04
7    resp_ip_bytes      0.0  4.870648e+06
8   dest_port_zeek      1.0  3.000000e+00
9       orig_bytes      0.0  2.188200e+05
10      resp_bytes      0.0  2.174312e+06
11   src_port_zeek      1.0  3.000000e+00


#### Step 1: Get the record closest to the centroid of the Attack Cluster
#### Step 2: Define the fields we want to change
#### Step 3: Populate the fields that we can't change with the record closest to the centroid of the Attack Cluster
#### Step 4: Define the Min and Max values of the fields we want to change (from what to what)
#### Step 5: Create the For Loops for every feature that will be changed
#### Step 6: For every different combination call the model
#### Step 7: If model misclassifies store combination in a list
#### Step 8: Analyze which fields had the highest impact on the model

In [27]:
import numpy as np
import pandas as pd
import math

from functools import reduce

from pyspark.sql import functions as F
from pyspark.sql.functions import mean, percentile_approx
from pyspark.ml.clustering import KMeans
from pyspark.ml.functions import vector_to_array

In [28]:
kmeans = KMeans(
    featuresCol="scaledFeatures",
    k=2,
    seed=200
)

model = kmeans.fit(df_scaled)

df_pred = (
    model
    .transform(df_scaled)
    .withColumnRenamed("prediction", "cluster")
)

centers = model.clusterCenters()

26/08/26 20:20:57 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
                                                                                

In [29]:
agg_df = df_pred.groupBy("cluster").agg(
    F.count("*").alias("total_count"),

    F.sum(
        F.when(F.col("label_binary") == True, 1).otherwise(0)
    ).alias("attacks"),

    F.sum(
        F.when(F.col("label_binary") == False, 1).otherwise(0)
    ).alias("normal_data")
)

attack_threshold = 0.5

attack_clusters_df = agg_df.filter(
    (
        F.col("attacks") /
        (F.col("attacks") + F.col("normal_data"))
    ) > attack_threshold
)

attack_clusters = [
    row["cluster"]
    for row in attack_clusters_df.select("cluster").collect()
]

print("Attack clusters:", attack_clusters)

[Stage 51:===========================================>              (3 + 1) / 4]

Attack clusters: [1]


In [30]:
assert len(attack_clusters) == 1

attack_cluster = attack_clusters[0]

normal_cluster = [
    c for c in range(2)
    if c not in attack_clusters
][0]

print("Attack cluster:", attack_cluster)
print("Normal cluster:", normal_cluster)

Attack cluster: 1
Normal cluster: 0


In [31]:
attack_center = np.array(
    centers[attack_cluster],
    dtype=float
)

print(attack_center)

[7.96956917e-04 3.08383376e-04 2.90630681e+00 1.35250037e-03
 1.91587788e-02 3.70243886e-04 4.09527004e-02 8.49655074e-05
 3.27008450e+00 4.36990148e-04 6.18958873e-05 4.83080426e+00]


In [32]:
attack_records = (
    df_pred
    .filter(
        (F.col("cluster") == attack_cluster) &
        (F.col("label_binary") == True)
    )
    .withColumn(
        "_scaled_array",
        vector_to_array("scaledFeatures")
    )
)

In [33]:
distance_terms = [
    F.pow(
        F.col("_scaled_array")[i] -
        F.lit(float(attack_center[i])),
        2
    )
    for i in range(len(columns_to_check))
]

distance_to_centroid = reduce(
    lambda a, b: a + b,
    distance_terms
)

attack_records = attack_records.withColumn(
    "distance_to_attack_centroid",
    distance_to_centroid
)

In [34]:
closest_attack_record = (
    attack_records
    .orderBy("distance_to_attack_centroid")
    .first()
)

print(
    "Squared distance:",
    closest_attack_record["distance_to_attack_centroid"]
)

26/08/26 20:35:54 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 54:===========================================>              (3 + 1) / 4]

Squared distance: 1.1042278230189377


In [35]:
baseline = {
    feature: closest_attack_record[feature]
    for feature in columns_to_check
}

baseline

{'duration': 0.0,
 'resp_pkts': 0,
 'conn_state': 3,
 'proto': 0,
 'orig_ip_bytes': 44,
 'missed_bytes': 0,
 'orig_pkts': 1,
 'resp_ip_bytes': 0,
 'dest_port_zeek': 2,
 'orig_bytes': 0,
 'resp_bytes': 0,
 'src_port_zeek': 3}

In [36]:
features_to_change = [
    "duration",
    "conn_state",
    "proto",
    "orig_ip_bytes",
    "orig_pkts",
    "dest_port_zeek",
    "orig_bytes",
    "src_port_zeek"
]

In [37]:
features_to_keep_fixed = [
    feature
    for feature in columns_to_check
    if feature not in features_to_change
]

print("Change:", features_to_change)
print("Fixed:", features_to_keep_fixed)

Change: ['duration', 'conn_state', 'proto', 'orig_ip_bytes', 'orig_pkts', 'dest_port_zeek', 'orig_bytes', 'src_port_zeek']
Fixed: ['resp_pkts', 'missed_bytes', 'resp_ip_bytes', 'resp_bytes']


In [38]:
fixed_values = {
    feature: baseline[feature]
    for feature in features_to_keep_fixed
}

fixed_values

{'resp_pkts': 0, 'missed_bytes': 0, 'resp_ip_bytes': 0, 'resp_bytes': 0}

In [39]:
categorical_features = [
    "conn_state",
    "proto",
    "dest_port_zeek",
    "src_port_zeek"
]

numeric_features = [
    "duration",
    "orig_ip_bytes",
    "orig_pkts",
    "orig_bytes"
]

grid_values = {}

# All observed categorical values
for feature in categorical_features:
    grid_values[feature] = [
        row[feature]
        for row in df_scaled.select(feature).distinct().orderBy(feature).collect()
    ]

# Representative numeric values
quantiles = [0.0, 0.10, 0.25, 0.50, 0.75, 0.90, 1.0]

for feature in numeric_features:
    values = df_scaled.approxQuantile(feature, quantiles, 0.001)
    values.append(float(baseline[feature]))
    grid_values[feature] = sorted(set(values))

for feature, values in grid_values.items():
    print(feature, values)

conn_state [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
proto [0, 1, 2]
dest_port_zeek [1, 2, 3]
src_port_zeek [1, 2, 3]
duration [0.0, 1.0013580322265625e-05, 1.621246337890625e-05, 0.0004611015319824219, 3560.855576992035]
orig_ip_bytes [0.0, 44.0, 134.0, 158.0, 180.0, 959736.0]
orig_pkts [0.0, 1.0, 2.0, 3.0, 10040.0]
orig_bytes [0.0, 78.0, 102.0, 218820.0]


In [40]:
from itertools import product

combinations = list(product(
    *[grid_values[f] for f in features_to_change]
))

print("Number of combinations:", len(combinations))

Number of combinations: 178200


In [41]:
candidate_records = []

for combination in combinations:

    record = baseline.copy()

    for feature, value in zip(features_to_change, combination):
        record[feature] = value

    candidate_records.append(record)

len(candidate_records)

178200

In [42]:
candidate_df = spark.createDataFrame(candidate_records)

candidate_df = candidate_df.select(*columns_to_check)

candidate_assembled = assembler.transform(candidate_df)

candidate_scaled = scaler_model.transform(candidate_assembled)

candidate_predictions = model.transform(candidate_scaled)

In [43]:
misclassified_df = candidate_predictions.filter(
    F.col("prediction") == normal_cluster
)

print(
    "Successful adversarial combinations:",
    misclassified_df.count()
)

26/09/04 15:20:36 WARN TaskSetManager: Stage 91 contains a task of very large size (2374 KiB). The maximum recommended task size is 1000 KiB.
[Stage 91:>                                                         (0 + 4) / 4]

Successful adversarial combinations: 143640


In [44]:
misclassified_df.select(
    *features_to_change,
    "prediction"
).show(50, truncate=False)

+--------+----------+-----+-------------+---------+--------------+----------+-------------+----------+
|duration|conn_state|proto|orig_ip_bytes|orig_pkts|dest_port_zeek|orig_bytes|src_port_zeek|prediction|
+--------+----------+-----+-------------+---------+--------------+----------+-------------+----------+
|0.0     |1         |0    |0.0          |0.0      |1             |0.0       |1            |0         |
|0.0     |1         |0    |0.0          |0.0      |1             |0.0       |2            |0         |
|0.0     |1         |0    |0.0          |0.0      |1             |0.0       |3            |0         |
|0.0     |1         |0    |0.0          |0.0      |1             |78.0      |1            |0         |
|0.0     |1         |0    |0.0          |0.0      |1             |78.0      |2            |0         |
|0.0     |1         |0    |0.0          |0.0      |1             |78.0      |3            |0         |
|0.0     |1         |0    |0.0          |0.0      |1             |102.0  

26/09/04 15:20:55 WARN TaskSetManager: Stage 94 contains a task of very large size (2374 KiB). The maximum recommended task size is 1000 KiB.


In [45]:
scored_df = candidate_predictions.withColumn(
    "evasion",
    F.when(
        F.col("prediction") == normal_cluster,
        1
    ).otherwise(0)
)

In [46]:
impact_results = {}

for feature in features_to_change:

    result = (
        scored_df
        .groupBy(feature)
        .agg(
            F.avg("evasion").alias("evasion_rate")
        )
        .orderBy(F.desc("evasion_rate"))
    )

    impact_results[feature] = result

    print("\nFEATURE:", feature)
    result.show()


FEATURE: duration


26/09/04 15:21:22 WARN TaskSetManager: Stage 95 contains a task of very large size (2374 KiB). The maximum recommended task size is 1000 KiB.
26/09/04 15:21:24 WARN TaskSetManager: Stage 98 contains a task of very large size (2374 KiB). The maximum recommended task size is 1000 KiB.


+--------------------+------------------+
|            duration|      evasion_rate|
+--------------------+------------------+
|   3560.855576992035|               1.0|
|                 0.0|0.7575757575757576|
|1.001358032226562...|0.7575757575757576|
|1.621246337890625E-5|0.7575757575757576|
|4.611015319824219E-4|0.7575757575757576|
+--------------------+------------------+


FEATURE: conn_state


+----------+------------------+
|conn_state|      evasion_rate|
+----------+------------------+
|         1|0.8666666666666667|
|         2|0.8518518518518519|
|         3| 0.837037037037037|
|         5|0.8222222222222222|
|         4|0.8222222222222222|
|         6|0.8074074074074075|
|         7|0.7925925925925926|
|         9|0.7777777777777778|
|         8|0.7777777777777778|
|        10| 0.762962962962963|
|        11|0.7481481481481481|
+----------+------------------+


FEATURE: proto


26/09/04 15:21:25 WARN TaskSetManager: Stage 101 contains a task of very large size (2374 KiB). The maximum recommended task size is 1000 KiB.
26/09/04 15:21:26 WARN TaskSetManager: Stage 104 contains a task of very large size (2374 KiB). The maximum recommended task size is 1000 KiB.


+-----+------------------+
|proto|      evasion_rate|
+-----+------------------+
|    2|0.9878787878787879|
|    1| 0.806060606060606|
|    0|0.6242424242424243|
+-----+------------------+


FEATURE: orig_ip_bytes


... (6 outputs left out)

26/09/04 15:21:30 WARN TaskSetManager: Stage 116 contains a task of very large size (2374 KiB). The maximum recommended task size is 1000 KiB.


+----------+------------------+
|orig_bytes|      evasion_rate|
+----------+------------------+
|  218820.0|               1.0|
|       0.0|0.7414141414141414|
|      78.0|0.7414141414141414|
|     102.0|0.7414141414141414|
+----------+------------------+


FEATURE: src_port_zeek


[Stage 116:============================>                            (2 + 2) / 4]

+-------------+------------------+
|src_port_zeek|      evasion_rate|
+-------------+------------------+
|            1|0.8181818181818182|
|            2| 0.806060606060606|
|            3| 0.793939393939394|
+-------------+------------------+



## Constrained Adversarial Robustness Analysis.
### Can we flip a representative attack using only ONE realistic feature change? If not, can TWO realistic changes do it?

In [48]:
# Use real attacks to define plausible adversarial values

attack_data = df_scaled.filter(
    F.col("label_binary") == True
)

In [49]:
from pyspark.sql.types import IntegerType, LongType

feature_schema = df_scaled.select(
    *columns_to_check
).schema

def cast_like_original(feature, value):

    dtype = feature_schema[feature].dataType

    if isinstance(dtype, (IntegerType, LongType)):
        return int(round(value))

    return float(value)

In [50]:
realistic_grid = {}

# Categorical features:
# only values that actually occur in attacks
for feature in categorical_features:

    realistic_grid[feature] = [
        row[feature]
        for row in (
            attack_data
            .select(feature)
            .distinct()
            .orderBy(feature)
            .collect()
        )
    ]

In [51]:
realistic_quantiles = [
    0.05,
    0.25,
    0.50,
    0.75,
    0.95
]

for feature in numeric_features:

    values = attack_data.approxQuantile(
        feature,
        realistic_quantiles,
        0.001
    )

    # Keep original value too
    values.append(baseline[feature])

    values = [
        cast_like_original(feature, x)
        for x in values
    ]

    realistic_grid[feature] = sorted(set(values))

In [52]:
for feature, values in realistic_grid.items():

    print(
        feature,
        "baseline =", baseline[feature],
        "tested =", values
    )

conn_state baseline = 3 tested = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
proto baseline = 0 tested = [0, 1]
dest_port_zeek baseline = 2 tested = [1, 2, 3]
src_port_zeek baseline = 3 tested = [1, 2, 3]
duration baseline = 0.0 tested = [0.0, 0.0006241798400878906]
orig_ip_bytes baseline = 44 tested = [40, 44, 195]
orig_pkts baseline = 1 tested = [1, 3]
orig_bytes baseline = 0 tested = [0, 114]


In [53]:
single_feature_records = []

for feature in features_to_change:

    for value in realistic_grid[feature]:

        # Skip original value
        if value == baseline[feature]:
            continue

        record = baseline.copy()

        record[feature] = value

        record["changed_feature"] = feature
        record["changed_value"] = str(value)

        single_feature_records.append(record)

print(
    "Single-feature experiments:",
    len(single_feature_records)
)

Single-feature experiments: 19


In [54]:
from pyspark.sql.types import StructField, StringType, StructType

single_schema = StructType(
    feature_schema.fields + [
        StructField(
            "changed_feature",
            StringType(),
            False
        ),
        StructField(
            "changed_value",
            StringType(),
            False
        )
    ]
)

single_df = spark.createDataFrame(
    single_feature_records,
    schema=single_schema
)

In [55]:
single_assembled = assembler.transform(
    single_df
)

single_scaled = scaler_model.transform(
    single_assembled
)

single_predictions = model.transform(
    single_scaled
)

In [56]:
single_results = single_predictions.withColumn(
    "evasion",
    F.when(
        F.col("prediction") == normal_cluster,
        1
    ).otherwise(0)
)

In [57]:
single_results.select(
    "changed_feature",
    "changed_value",
    "prediction",
    "evasion"
).show(
    100,
    truncate=False
)

+---------------+---------------------+----------+-------+
|changed_feature|changed_value        |prediction|evasion|
+---------------+---------------------+----------+-------+
|duration       |0.0006241798400878906|1         |0      |
|conn_state     |1                    |1         |0      |
|conn_state     |2                    |1         |0      |
|conn_state     |4                    |1         |0      |
|conn_state     |5                    |1         |0      |
|conn_state     |6                    |1         |0      |
|conn_state     |7                    |1         |0      |
|conn_state     |8                    |1         |0      |
|conn_state     |9                    |1         |0      |
|conn_state     |10                   |1         |0      |
|proto          |1                    |0         |1      |
|orig_ip_bytes  |40                   |1         |0      |
|orig_ip_bytes  |195                  |1         |0      |
|orig_pkts      |3                    |1         |0     

In [58]:
successful_single = single_results.filter(
    F.col("evasion") == 1
)

print(
    "Successful ONE-feature evasions:",
    successful_single.count()
)

Successful ONE-feature evasions: 1


In [59]:
successful_single.select(
    "changed_feature",
    "changed_value",
    *features_to_change
).show(
    100,
    truncate=False
)

+---------------+-------------+--------+----------+-----+-------------+---------+--------------+----------+-------------+
|changed_feature|changed_value|duration|conn_state|proto|orig_ip_bytes|orig_pkts|dest_port_zeek|orig_bytes|src_port_zeek|
+---------------+-------------+--------+----------+-----+-------------+---------+--------------+----------+-------------+
|proto          |1            |0.0     |3         |1    |44           |1        |2             |0         |3            |
+---------------+-------------+--------+----------+-----+-------------+---------+--------------+----------+-------------+



In [60]:
single_summary = (
    single_results
    .groupBy("changed_feature")
    .agg(
        F.count("*").alias("tested"),
        F.sum("evasion").alias("successful"),
        F.avg("evasion").alias("evasion_rate")
    )
    .orderBy(
        F.desc("evasion_rate")
    )
)

single_summary.show(
    truncate=False
)

+---------------+------+----------+------------+
|changed_feature|tested|successful|evasion_rate|
+---------------+------+----------+------------+
|proto          |1     |1         |1.0         |
|duration       |1     |0         |0.0         |
|conn_state     |9     |0         |0.0         |
|orig_ip_bytes  |2     |0         |0.0         |
|orig_pkts      |1     |0         |0.0         |
|orig_bytes     |1     |0         |0.0         |
|src_port_zeek  |2     |0         |0.0         |
|dest_port_zeek |2     |0         |0.0         |
+---------------+------+----------+------------+



In [61]:
from itertools import combinations, product

feature_pairs = list(
    combinations(
        features_to_change,
        2
    )
)

print(
    "Number of feature pairs:",
    len(feature_pairs)
)

Number of feature pairs: 28


In [62]:
two_feature_records = []

for feature_1, feature_2 in feature_pairs:

    for value_1, value_2 in product(
        realistic_grid[feature_1],
        realistic_grid[feature_2]
    ):

        # Both must actually change
        if value_1 == baseline[feature_1]:
            continue

        if value_2 == baseline[feature_2]:
            continue

        record = baseline.copy()

        record[feature_1] = value_1
        record[feature_2] = value_2

        record["changed_features"] = (
            feature_1 + " + " + feature_2
        )

        two_feature_records.append(record)

print(
    "Two-feature experiments:",
    len(two_feature_records)
)

Two-feature experiments: 132


In [63]:
two_schema = StructType(
    feature_schema.fields + [
        StructField(
            "changed_features",
            StringType(),
            False
        )
    ]
)

In [64]:
two_df = spark.createDataFrame(
    two_feature_records,
    schema=two_schema
)

two_assembled = assembler.transform(
    two_df
)

two_scaled = scaler_model.transform(
    two_assembled
)

two_predictions = model.transform(
    two_scaled
)

In [65]:
two_results = two_predictions.withColumn(
    "evasion",
    F.when(
        F.col("prediction") == normal_cluster,
        1
    ).otherwise(0)
)

In [66]:
pair_summary = (
    two_results
    .groupBy("changed_features")
    .agg(
        F.count("*").alias("tested"),
        F.sum("evasion").alias("successful"),
        F.avg("evasion").alias("evasion_rate")
    )
    .orderBy(
        F.desc("evasion_rate")
    )
)

pair_summary.show(
    50,
    truncate=False
)

+------------------------------+------+----------+-------------------+
|changed_features              |tested|successful|evasion_rate       |
+------------------------------+------+----------+-------------------+
|duration + proto              |1     |1         |1.0                |
|proto + src_port_zeek         |2     |2         |1.0                |
|proto + orig_ip_bytes         |2     |2         |1.0                |
|proto + orig_pkts             |1     |1         |1.0                |
|proto + orig_bytes            |1     |1         |1.0                |
|proto + dest_port_zeek        |2     |1         |0.5                |
|conn_state + proto            |9     |4         |0.4444444444444444 |
|dest_port_zeek + src_port_zeek|4     |1         |0.25               |
|conn_state + dest_port_zeek   |18    |1         |0.05555555555555555|
|duration + orig_ip_bytes      |2     |0         |0.0                |
|duration + src_port_zeek      |2     |0         |0.0                |
|durat

In [67]:
successful_pairs = two_results.filter(
    F.col("evasion") == 1
)

print(
    "Successful TWO-feature evasions:",
    successful_pairs.count()
)

Successful TWO-feature evasions: 14


In [68]:
successful_pairs.select(
    "changed_features",
    *features_to_change,
    "prediction"
).show(
    100,
    truncate=False
)

+------------------------------+--------------------+----------+-----+-------------+---------+--------------+----------+-------------+----------+
|changed_features              |duration            |conn_state|proto|orig_ip_bytes|orig_pkts|dest_port_zeek|orig_bytes|src_port_zeek|prediction|
+------------------------------+--------------------+----------+-----+-------------+---------+--------------+----------+-------------+----------+
|duration + proto              |6.241798400878906E-4|3         |1    |44           |1        |2             |0         |3            |0         |
|conn_state + proto            |0.0                 |1         |1    |44           |1        |2             |0         |3            |0         |
|conn_state + proto            |0.0                 |2         |1    |44           |1        |2             |0         |3            |0         |
|conn_state + proto            |0.0                 |4         |1    |44           |1        |2             |0         |3   